In [ ]:
## Main Loop
n_steps = 100
mutation_steps = 1
use_SMW = True

seed_sequence = tf.argmax(x_train[1300], axis=1).numpy().tolist()
if use_SMW: 
    # Start from a seed sequence in the training data (convert one-hot to integer indices)
    smw = SMW(nucleotides=[0, 1, 2, 3])


# Flatten x_train for comparison
x_train_flat = x_train.reshape(x_train.shape[0], flat_shape)

df_all_values = pd.DataFrame({'mutation': [], 'predicted_score': [], 'real_score': [], 'error': []})

def calculate_error(predicted_scores, real_scores):
    errors = []
    for pred, real in zip(predicted_scores, real_scores):
        error = abs(pred - real)
        errors.append(error)
    return errors

score_prev = -1
seen_sequences = x_train_flat

for i in range(n_steps):
    values = {'mutation': [], 'predicted_score': []}
    if not use_SMW:
        mutations = generate_mutations(seed_sequence, mutation_steps)
        # one-hot encode mutations
        mutations = [tf.one_hot(mut, depth=4).numpy().reshape(1, flat_shape) for mut in mutations]
    else: 
        mutations = smw.walk(seed_sequence, mutation_steps)

    # check if mutations are in train data
    for idx, mut in enumerate(mutations):
        # Check if mutation is NOT in training data (try with complete dataset)
        already_seen = np.any(np.all(seen_sequences == mut, axis=1))
        if not already_seen:
            # If not in training data, add to list
            mutations_score = rf.predict(mut)
            values['mutation'].append(mut)
            values['predicted_score'].append(mutations_score[0])
        
    
    # Select best mutation -> highest value
    # sort values by predicted score -> highest score first
    values_df = pd.DataFrame(values)
    values_df_sorted = values_df.sort_values(by='predicted_score', ascending=False)
    # keep only top 1
    values_df_sorted = values_df_sorted.head(1)
    # append to all values dataframe for later analysis
    df_all_values = pd.concat([df_all_values, values_df_sorted], ignore_index=True)

    # update seed sequence to best mutation -> highes predicted score
    if len(values_df_sorted) == 0:
        print(f'Step {i+1}/{n_steps}, No new mutations found.')
        continue

    # update seed sequence only if score is higher than previous best
    value_new =values_df_sorted.iloc[0]["predicted_score"]
    if value_new > score_prev:
        score_prev = value_new
        seed_sequence_one_hot = values_df_sorted.iloc[0]['mutation']
        seed_sequence = tf.argmax(tf.reshape(seed_sequence_one_hot, (seed_sequence_one_hot.shape[1]//4, 4)), axis=1).numpy().tolist()
        print(f'Step {i+1}/{n_steps}, Best predicted score: {values_df_sorted.iloc[0]["predicted_score"]}')
    else:
        print(f'Step {i+1}/{n_steps}, No better mutation found. Keeping previous seed sequence.')    
    
    # make shure not to check seen sequences again
    if len(values['mutation']) > 0:
        new_mutations = np.vstack(values['mutation'])
        seen_sequences = np.vstack([seen_sequences, new_mutations])
    print(f'New seed sequence: {seed_sequence}')


In [ ]:
#lookup real scores and calculate error
for index, row in df_all_values.iterrows():
    real_score = oracle_lookup(row['mutation'])
    df_all_values.at[index, 'real_score'] = real_score
    error = abs(row['predicted_score'] - real_score)
    df_all_values.at[index, 'error'] = error

print(df_all_values.head(10))

In [ ]:
import matplotlib.pyplot as plt


real_scores = df_all_values['real_score']
predicted_scores = df_all_values['predicted_score']
# Plot real vs predicted scores
plt.figure(figsize=(12, 6))
# Downsample to every 10th point for better visibility
step = 1
indices = range(0, len(real_scores), step)
real_scores_sampled = real_scores.iloc[::step]
predicted_scores_sampled = predicted_scores.iloc[::step]
plt.plot(range(len(real_scores_sampled)), real_scores_sampled, label='Real Binding Scores', alpha=0.7, marker='o', markersize=3)
plt.plot(range(len(predicted_scores_sampled)), predicted_scores_sampled, label='Predicted Scores', alpha=0.7, marker='o', markersize=3)
plt.xlabel('Timestep')
plt.ylabel('Binding Score')
plt.title('Real vs Predicted Binding Scores for New Mutations')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Calculate absolute error for each mutation
errors = [abs(real - pred) for real, pred in zip(real_scores, predicted_scores)]

# Plot error over mutation index
plt.figure(figsize=(10, 6))
plt.plot(range(len(errors)), errors, marker='o', markersize=2, alpha=0.7)
plt.xlabel('Mutation Index (sorted by predicted score)')
plt.ylabel('Absolute Error (|Real - Predicted|)')
plt.title('Prediction Error Over Mutations')
plt.tight_layout()
plt.show()

# Also plot a rolling average to see the trend more clearly
window_size = 20
if len(errors) >= window_size:
    rolling_errors = pd.Series(errors).rolling(window=window_size).mean()
    plt.figure(figsize=(10, 6))
    plt.plot(range(len(errors)), errors, alpha=0.3, label='Individual Errors')
    plt.plot(range(len(rolling_errors)), rolling_errors, color='red', linewidth=2, label=f'Rolling Average (window={window_size})')
    plt.xlabel('Mutation Index (sorted by predicted score)')
    plt.ylabel('Absolute Error')
    plt.title('Prediction Error with Rolling Average')
    plt.legend()
    plt.tight_layout()
    plt.show()